# GPCN-ViT: Graph Patch Correlation Network + Vision Transformer
**Breast-cancer histopathology classification on BreakHis (binary: benign vs malignant)**

Full publishable pipeline:
- **Backbone:** Phikon pathology foundation model (ViT-B/16, TCGA H&E); falls back to ImageNet ViT.
- **GPCN** integrated *with* ViT attention (not replacing it) + multi-scale graph pyramid.
- **Training:** class-weighted focal loss, weight EMA, warmup→cosine, discriminative LR, AUC model-selection.
- **Inference:** Youden decision-threshold tuning (val→test) + flip-TTA.
- **Validity:** patient-level split (no leakage), full clinical metrics + calibration + uncertainty.

> **Crash / timeout safe:** every training cell **auto-resumes** from its last checkpoint — just re-run the cell after a Colab disconnect and it continues where it stopped.

### Run order
`1–4` setup → `5` smoke test → `6` single magnification **or** `9` all magnifications → `10` multi-magnification fusion → `11` publication tables → `12` GPCN ablation.

In [ ]:
# ── CELL 1 ── GPU check + dependency installation

import subprocess, sys, os

def _run(cmd, desc=''):
    if desc:
        print(f'  {desc}...', end=' ', flush=True)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if desc:
        print('done' if result.returncode == 0 else f'WARN ({result.returncode})')
    return result

print('=' * 70)
print('  STEP 1/6 — Environment Setup')
print('=' * 70)

# GPU info
gpu_result = _run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
if gpu_result.returncode == 0 and gpu_result.stdout.strip():
    print(f'  GPU: {gpu_result.stdout.strip()}')
else:
    print('  WARNING: No GPU detected — training will be very slow on CPU')

# Detect PyTorch version already installed in Colab
try:
    import torch
    torch_ver = torch.__version__
    cuda_ver  = torch.version.cuda or 'none'
    print(f'  PyTorch: {torch_ver}   CUDA: {cuda_ver}')
except ImportError:
    torch_ver = '2.1.0'
    print('  PyTorch not found, installing 2.1.0+cu118...')
    _run('pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
         'Installing PyTorch')

# Determine PyG wheel URL from torch version
torch_short = '.'.join(torch_ver.split('.')[:2])  # e.g. '2.1'
pyg_url = f'https://data.pyg.org/whl/torch-{torch_short}.0+cu118.html'

packages_basic = [
    ('timm',                   'timm (ViT models)'),
    ('wandb',                  'Weights & Biases'),
    ('opencv-python-headless', 'OpenCV'),
    ('scikit-learn',           'scikit-learn'),
    ('scikit-image',           'scikit-image'),
    ('tqdm',                   'tqdm'),
    ('seaborn',                'Seaborn'),
    ('staintools',             'StainTools (stain normalisation)'),
]

for pkg, desc in packages_basic:
    _run(f'pip install -q {pkg}', desc)

# PyTorch Geometric
print('  Installing PyTorch Geometric...', end=' ', flush=True)
r1 = _run('pip install -q torch-geometric')
r2 = _run(f'pip install -q pyg-lib torch-scatter torch-sparse -f {pyg_url}')
print('done' if (r1.returncode == 0 and r2.returncode == 0) else 'WARN (check manually)')

# Verify critical imports
failed = []
for mod in ['torch', 'timm', 'torch_geometric', 'sklearn', 'cv2']:
    try:
        __import__(mod)
    except ImportError:
        failed.append(mod)

if failed:
    print(f'\n  FAILED imports: {failed}')
    print('  Please install manually before continuing.')
else:
    print('\n  All critical packages installed successfully!')

print('=' * 70)

In [ ]:
# ── CELL 2 ── Mount Google Drive + configure paths

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Update DATASET_PATH to where you stored BreaKHis_v1 inside your Drive.
DATASET_PATH = '/content/drive/MyDrive/MY RESEARCH/BreaKHis_v1'
OUTPUT_PATH  = '/content/drive/MyDrive/MY RESEARCH/gpcn_vit_outputs'
# BACH (ICIAR-2018) Part A microscopy images, for second-dataset training
# + external validation. Point this at the folder holding the class
# sub-folders (Normal/Benign/InSitu/Invasive), e.g. .../ICIAR2018_BACH_Challenge/Photos
BACH_PATH    = '/content/drive/MyDrive/MY RESEARCH/BACH/Photos'
# ─────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path

os.makedirs(OUTPUT_PATH, exist_ok=True)

print('=' * 70)
print('  STEP 2/6 — Google Drive')
print('=' * 70)
print(f'  Drive mounted at /content/drive')

if Path(DATASET_PATH).exists():
    print(f'  Dataset : {DATASET_PATH}  [FOUND]')
else:
    print(f'  Dataset : {DATASET_PATH}  [NOT FOUND]')
    print('  Please update DATASET_PATH above to the actual BreaKHis_v1 location.')

print(f'  Outputs : {OUTPUT_PATH}')
print('=' * 70)

In [ ]:
# ── CELL 3 ── Copy project files from Google Drive
# Files are imported from: My Drive / MY RESEARCH / Research /

import shutil
from pathlib import Path

# Path to your project files on Google Drive
DRIVE_CODE_PATH = '/content/drive/MyDrive/MY RESEARCH/Research'

REQUIRED_FILES = [
    'config.py', 'model.py', 'gpcn_layer.py', 'knn_builder.py',
    'dataset.py', 'augmentation.py', 'gan_augmentation.py',
    'losses.py', 'metrics.py', 'calibration.py',
    'trainer.py', 'train.py', 'utils.py', 'visualization.py',
    'multimag.py', 'experiments.py', 'single_mag_baseline.py', 'bach.py',
    'kfold_cv.py', 'calibration_report.py', 'stats_tests.py',
    'fusion_baselines.py',
]

print('=' * 70)
print('  STEP 3/6 — Project Files')
print(f'  Source  : {DRIVE_CODE_PATH}')
print(f'  Target  : /content/')
print('=' * 70)

drive_dir = Path(DRIVE_CODE_PATH)
if not drive_dir.exists():
    print(f'  ERROR: Drive folder not found: {DRIVE_CODE_PATH}')
    print('  Check the path and re-run this cell.')
else:
    copied, missing = [], []
    for f in REQUIRED_FILES:
        src = drive_dir / f
        if src.exists():
            shutil.copy(str(src), f'/content/{f}')
            copied.append(f)
        else:
            missing.append(f)

    print(f'  Copied  ({len(copied)}/{len(REQUIRED_FILES)}):')
    for f in copied:
        print(f'    {f}  [OK]')

    if missing:
        print(f'\n  Missing ({len(missing)}):')
        for f in missing:
            print(f'    {f}  [NOT FOUND in Drive]')
        print('\n  Tip: upload the missing files to your Drive folder and re-run.')
    else:
        print('\n  All required files copied to /content/')

print('=' * 70)

In [ ]:
# ── CELL 4 ── Verify imports
import torch, timm, sklearn, numpy, PIL
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

import config, dataset, model, trainer, losses, metrics, utils, augmentation
import gpcn_layer, knn_builder, multimag, experiments
import bach, single_mag_baseline, kfold_cv, calibration_report, stats_tests
from config import get_default_config

c = get_default_config()
print('Default backbone        :', c.model.backbone)
print('Threshold tuning        :', c.validation.optimize_threshold, '(', c.validation.threshold_mode, ')')
print('TTA / EMA / auto_resume  :', c.validation.use_tta, '/', c.training.use_ema, '/', c.training.auto_resume)
print('✓ all modules import OK')

In [ ]:
# ── CELL 5 ── Quick smoke test (5 epochs, ~10 min on T4)
# Run this first to confirm everything works before committing to full training.

import torch
from config import get_quick_test_config
from dataset import create_dataloaders
from model import create_model
from trainer import Trainer
from utils import set_seed, count_parameters

config = get_quick_test_config()
config.data.data_root          = DATASET_PATH
config.data.save_dir           = OUTPUT_PATH
config.experiment_name         = 'quick_test'
config.logging.use_wandb       = False
config.logging.use_tensorboard = False
config.device                  = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(42, deterministic=False)

print('=' * 70)
print('  QUICK TEST — 5 Epochs (smoke test, ~10 min on T4)')
print('=' * 70)
print(f'  Device        : {config.device.upper()}')
print(f'  Magnification : {config.data.train_magnification}')
print(f'  Batch size    : {config.data.batch_size}')

print('\n  Loading dataset...')
train_loader, val_loader, test_loader, info = create_dataloaders(config)
print(f'  Train: {info["train_samples"]}  Val: {info["val_samples"]}  Test: {info["test_samples"]}')

print('\n  Building model...')
model = create_model(config)
pc    = count_parameters(model)
print(f'  Parameters: {pc["total"]:,}  Trainable: {pc["trainable"]:,}')

print('\n  Training...')
device  = torch.device(config.device)
trainer = Trainer(
    model=model, config=config,
    train_loader=train_loader, val_loader=val_loader,
    test_loader=test_loader,
    class_weights=info['class_weights'].to(device),
)

results = trainer.train()

print('\n' + '=' * 70)
print('  QUICK TEST COMPLETE')
print(f'  Best Val Accuracy : {results["best_val_acc"]:.4f}')
print(f'  Best Val AUC      : {results["best_val_auc"]:.4f}')
print('=' * 70)

In [ ]:
# ── CELL 6 ── Full training — single magnification
#
# Defaults already enabled: Phikon backbone + class-weighted focal + EMA +
# warmup→cosine + discriminative LR + AUC selection + Youden threshold tuning + TTA.
#
# CRASH / TIMEOUT SAFE: training auto-resumes from last_checkpoint.pth.
# Just re-run this cell after a disconnect — it continues where it stopped.

import torch
from config import get_default_config
from dataset import create_augmented_dataloaders
from model import create_model
from trainer import Trainer
from utils import set_seed, count_parameters

MAGNIFICATION = '40X'    # '40X' | '100X' | '200X' | '400X'
NUM_EPOCHS    = 50
BATCH_SIZE    = 16       # reduce to 8 on T4 if OOM
SEED          = 42

config = get_default_config()
config.data.data_root           = DATASET_PATH
config.data.save_dir            = OUTPUT_PATH
config.data.train_magnification = MAGNIFICATION
config.data.batch_size          = BATCH_SIZE
config.training.num_epochs      = NUM_EPOCHS
config.logging.use_wandb        = False
config.logging.use_tensorboard  = True
config.experiment_name          = f'gpcn_vit_{MAGNIFICATION}_{NUM_EPOCHS}ep'
config.device                   = 'cuda' if torch.cuda.is_available() else 'cpu'
config.seed                     = SEED

set_seed(SEED, deterministic=False)
device = torch.device(config.device)

print('=' * 70)
print(f'  FULL TRAINING — {NUM_EPOCHS} epochs @ {MAGNIFICATION}')
print('=' * 70)
print(f'  Backbone : {config.model.backbone}')
print(f'  Threshold: {config.validation.optimize_threshold} ({config.validation.threshold_mode}) | TTA: {config.validation.use_tta}')

train_loader, val_loader, test_loader, info = create_augmented_dataloaders(config, device)
print(f'  Train: {info["train_samples"]}  Val: {info["val_samples"]}  Test: {info["test_samples"]}')

model = create_model(config).to(device)
print(f'  Params: {count_parameters(model)["total"]:,}')

trainer = Trainer(
    model=model, config=config,
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    class_weights=info['class_weights'].to(device),
)

# trainer.train() auto-resumes from last_checkpoint.pth if it exists.
results = trainer.train()

print('\n' + '=' * 70)
print(f'  Best Val AUC : {results["best_val_auc"]:.4f}  | Best epoch: {results["best_epoch"] + 1}')
print('  Test set (threshold-tuned):')
for k, v in sorted(results['test_metrics'].items()):
    print(f'    {k:<35} {v:.4f}')
print('=' * 70)

In [ ]:
# ── CELL 7 ── Evaluation & visualisation

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve
from metrics import evaluate_model

device = torch.device(config.device)

# Load best checkpoint
best_ckpt = f'{OUTPUT_PATH}/{config.experiment_name}/checkpoints/best_model.pth'
ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
base = model.base_model if hasattr(model, 'base_model') else model
base.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded checkpoint: {best_ckpt}')

test_metrics, y_true, y_pred, y_probs = evaluate_model(
    model, test_loader, device,
    use_uncertainty=config.model.use_uncertainty
)

print(f"\n{'Metric':<35} {'Value':>10}")
print('─' * 47)
for k, v in sorted(test_metrics.items()):
    print(f'  {k:<33} {v:>10.4f}')

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'], ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_title('GPCN-ViT — Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_PATH}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#E84040', lw=3, label=f'GPCN-ViT (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_PATH}/roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Precision-Recall Curve ────────────────────────────────────────────────────
precision, recall, _ = precision_recall_curve(y_true, y_probs[:, 1])
pr_auc = auc(recall, precision)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, color='#6A5ACD', lw=3, label=f'GPCN-ViT (AUC = {pr_auc:.4f})')
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curve', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_PATH}/pr_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Reliability Diagram (Calibration) ────────────────────────────────────────
n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
conf, acc = [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (y_probs[:, 1] >= lo) & (y_probs[:, 1] < hi)
    if mask.sum() > 0:
        conf.append(y_probs[:, 1][mask].mean())
        acc.append((y_true[mask] == 1).mean())
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(conf, acc, 'o-', color='#E84040', ms=8, lw=2, label='GPCN-ViT')
ax.set_xlabel('Confidence', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Reliability Diagram (Calibration)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_PATH}/calibration.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\nPlots saved to: {OUTPUT_PATH}')

In [ ]:
# ── CELL 7b ── Reliability diagram + temperature scaling (calibration)
#
# A clinical model's probabilities must be trustworthy: when it says 0.9 it
# should be right ~90% of the time. This fits a single temperature T on the
# VALIDATION set (never the test set), then shows the TEST-set reliability
# diagram and ECE/MCE BEFORE vs AFTER scaling. Reuses the best model + loaders
# already in memory from CELL 6 / CELL 7.

from calibration_report import temperature_calibration_report

cal = temperature_calibration_report(
    model, val_loader, test_loader, device,
    save_path=f'{OUTPUT_PATH}/{config.experiment_name}_reliability.png',
    class_names=['Benign', 'Malignant'],
)
print('\nCalibration (test set):')
print(f"  Temperature T        : {cal['temperature']:.4f}")
print(f"  ECE  before -> after : {cal['ece_before']:.4f} -> {cal['ece_after']:.4f}  (delta {cal['ece_reduction']:+.4f})")
print(f"  MCE  before -> after : {cal['mce_before']:.4f} -> {cal['mce_after']:.4f}  (delta {cal['mce_reduction']:+.4f})")

# STANDALONE (without re-running CELL 6/7): load a saved checkpoint instead --
# from calibration_report import report_from_checkpoint
# cal = report_from_checkpoint(
#     config, f'{OUTPUT_PATH}/{config.experiment_name}/checkpoints/best_model.pth',
#     save_path=f'{OUTPUT_PATH}/reliability.png')
#
# NOTE: for the BACH multi-class model the SAME function auto-switches to a
# top-label reliability diagram -- pass that model + its val/test loaders.


In [ ]:
# ── CELL 8 ── Download results to your local machine

from google.colab import files
from pathlib import Path

to_download = [
    f'{OUTPUT_PATH}/{config.experiment_name}/checkpoints/best_model.pth',
    f'{OUTPUT_PATH}/confusion_matrix.png',
    f'{OUTPUT_PATH}/roc_curve.png',
    f'{OUTPUT_PATH}/pr_curve.png',
    f'{OUTPUT_PATH}/calibration.png',
]

for path in to_download:
    if Path(path).exists():
        try:
            files.download(path)
            print(f'  Downloaded : {Path(path).name}')
        except Exception as exc:
            print(f'  Could not auto-download {Path(path).name}: {exc}')
    else:
        print(f'  Not found (skipped): {path}')

print('\nTip: if auto-download fails, use the Files panel in Colab (folder icon on left).')

In [ ]:
# ── CELL 9 ── All magnifications, sequential (40X → 100X → 200X → 400X)
#
# CRASH / TIMEOUT SAFE:
#   • Completed magnifications are skipped (training_progress.json).
#   • A partially-trained magnification auto-resumes from its last_checkpoint.pth.
#   • Just re-run this cell — it figures out where to continue.

import json, torch
from pathlib import Path
from config import get_default_config
from dataset import create_augmented_dataloaders
from model import create_model
from trainer import Trainer
from utils import set_seed, count_parameters

NUM_EPOCHS_MULTI = 50
BATCH_SIZE_MULTI = 16     # reduce to 8 on T4 if OOM
BASE_EXP_NAME    = 'gpcn_vit_all_mags'
SEED             = 42

ALL_MAGS  = ['40X', '100X', '200X', '400X']
PROG_FILE = Path(OUTPUT_PATH) / BASE_EXP_NAME / 'training_progress.json'
PROG_FILE.parent.mkdir(parents=True, exist_ok=True)

if PROG_FILE.exists():
    progress = json.load(open(PROG_FILE))
    print('Already completed:', list(progress.get('completed', {}).keys()), '\n')
else:
    progress = {'completed': {}}
    print('No progress file — starting from scratch.\n')

for mag in ALL_MAGS:
    if mag in progress['completed']:
        print(f'[{mag}] already done — skipping.')
        continue

    print('=' * 70 + f'\n  MAGNIFICATION: {mag}\n' + '=' * 70)
    cfg = get_default_config()
    cfg.data.data_root           = DATASET_PATH
    cfg.data.save_dir            = OUTPUT_PATH
    cfg.data.train_magnification = mag
    cfg.data.batch_size          = BATCH_SIZE_MULTI
    cfg.training.num_epochs      = NUM_EPOCHS_MULTI
    cfg.logging.use_wandb        = False
    cfg.experiment_name          = f'{BASE_EXP_NAME}_{mag}'
    cfg.device                   = 'cuda' if torch.cuda.is_available() else 'cpu'
    cfg.seed                     = SEED

    set_seed(SEED, deterministic=False)
    device = torch.device(cfg.device)

    train_loader, val_loader, test_loader, info = create_augmented_dataloaders(cfg, device)
    m = create_model(cfg).to(device)
    t = Trainer(model=m, config=cfg,
                train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
                class_weights=info['class_weights'].to(device))

    run_result = t.train()   # auto-resumes a partial run automatically

    progress['completed'][mag] = {
        'best_val_auc': float(run_result['best_val_auc']),
        'best_epoch': int(run_result['best_epoch']),
        'test_metrics': {k: float(v) for k, v in run_result['test_metrics'].items()},
    }
    json.dump(progress, open(PROG_FILE, 'w'), indent=2)
    print(f'[{mag}] done — AUC {run_result["best_val_auc"]:.4f}. Progress saved.\n')

# ── Publication tables from completed magnifications ──────────────────────────
from experiments import build_results_table, build_comparison_table
recs = [{'magnification': m, 'backbone': 'phikon', 'use_gpcn': True,
         'metrics': progress['completed'][m]['test_metrics']}
        for m in ALL_MAGS if m in progress['completed']]
print('\n' + build_results_table(recs))
print('\n' + build_comparison_table(recs))

In [ ]:
# ── CELL 10 ── Multi-magnification FUSION model (MultiMagnificationGPCNViT)
#
# Fuses all four magnifications of the SAME slide via cross-magnification
# attention + learnable magnification weights. Valid multi-scale fusion with
# patient-level splitting (no leakage). NOT pixel-registered (feature fusion).
#
# CRASH / TIMEOUT SAFE: auto-resumes from last_multimag.pth — re-run to continue.

import torch
from config import get_default_config
from multimag import train_multimag

config = get_default_config()
config.data.data_root      = DATASET_PATH
config.data.save_dir       = OUTPUT_PATH
config.data.batch_size     = 8       # 4x ViT-B forwards/sample — keep small on T4
config.training.num_epochs = 50
config.experiment_name     = 'gpcn_multimag'
config.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

result = train_multimag(config)   # auto-resumes if interrupted

print('\nMulti-magnification fusion — test metrics (threshold-tuned):')
for k, v in sorted(result['test_metrics'].items()):
    print(f'  {k:<35} {v:.4f}')

In [ ]:
# ── CELL 10b ── Fair fused-vs-single-magnification baseline (identical slide-level test set)
#
# Loads best_multimag.pth and RE-EVALUATES only — no retraining. Reports each
# single magnification on the SAME slide-level test set as the fused model, so the
# comparison is apples-to-apples (the per-IMAGE per-mag table is a different unit).
# Run this AFTER CELL 10; it reuses the `config` defined there.

import importlib, single_mag_baseline
importlib.reload(single_mag_baseline)          # pick up the file without a kernel restart
from single_mag_baseline import run_single_mag_baseline

baseline = run_single_mag_baseline(config)     # prints the comparison table

fused      = baseline['fused']
single_avg = baseline['single_avg']
print('\nHeadline (identical slide-level test set):')
print(f"  accuracy  fused {fused['accuracy']:.4f}  vs  single-mag avg {single_avg['accuracy']:.4f}"
      f"   (delta {fused['accuracy']-single_avg['accuracy']:+.4f})")
print(f"  MCC       fused {fused['mcc']:.4f}  vs  single-mag avg {single_avg['mcc']:.4f}"
      f"   (delta {fused['mcc']-single_avg['mcc']:+.4f})")


In [ ]:
# ── CELL 10c ── Naive-fusion baselines (defends the fusion MECHANISM)
#
# Isolates whether the cross-magnification ATTENTION beats dumb fusion of the same
# four streams. Two parts (fills Table `tab:fusion-baselines`):
#   (A) FREE  — post-hoc late fusion (avg-prob / majority-vote) from the ALREADY
#               trained fusion model. No retraining. McNemar p + Wilson CI.
#   (B) CLEAN — retrain from scratch with the attention REMOVED (mean / concat /
#               none) under the identical recipe. This is the primary comparison.
# Run this AFTER CELL 10 (needs best_multimag.pth). Reuses `config` from CELL 10.

import importlib, torch, multimag, fusion_baselines
importlib.reload(fusion_baselines)   # pick up the file without a kernel restart
from fusion_baselines import run_posthoc_fusion_baselines, build_fusion_model

# ── (A) FREE post-hoc baselines — no training ────────────────────────────────
print('=' * 78)
print('  (A) POST-HOC late-fusion baselines  (no retrain)')
print('=' * 78)
posthoc = run_posthoc_fusion_baselines(config)   # loads best_multimag.pth, prints table

# ── (B) CLEAN from-scratch fusion variants ───────────────────────────────────
# EXPENSIVE: each mode is a full multi-magnification training run (~like CELL 10).
# CRASH-SAFE: each arm has its own experiment_name and auto-resumes from its
# last_multimag.pth — just re-run this cell to continue. Set to [] to skip.
FROM_SCRATCH_MODES = ['mean', 'concat', 'none']

# Monkey-patch so the existing train loop builds each ablation arm unchanged.
_orig_builder = multimag.build_multimag_model
multimag.build_multimag_model = build_fusion_model

fusion_results = {'attention': posthoc['attention']}
try:
    for mode in FROM_SCRATCH_MODES:
        print('\n' + '=' * 78)
        print(f'  (B) FROM-SCRATCH fusion arm — mode = {mode!r}')
        print('=' * 78)
        from config import get_default_config
        cfg = get_default_config()
        cfg.data.data_root      = DATASET_PATH
        cfg.data.save_dir       = OUTPUT_PATH
        cfg.data.batch_size     = 8
        cfg.training.num_epochs = 50
        cfg.model.fusion_mode   = mode
        cfg.experiment_name     = f'fusion_{mode}'
        cfg.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

        arm = train_multimag(cfg)   # auto-resumes if interrupted
        fusion_results[mode] = arm['test_metrics']
        print(f'  [{mode}] done — test metrics:')
        for k, v in sorted(arm['test_metrics'].items()):
            print(f'    {k:<28} {v:.4f}')
finally:
    multimag.build_multimag_model = _orig_builder   # always restore

# ── Summary table: attention (ours) vs each naive fusion ─────────────────────
ROWS = ['accuracy', 'auc_roc', 'mcc', 'f1_class1', 'sensitivity', 'specificity']
order = ['attention'] + FROM_SCRATCH_MODES
print('\n' + '=' * 78)
print('  FUSION-MECHANISM ABLATION — from-scratch arms, identical recipe')
print('=' * 78)
print(f"{'metric':<14}" + ''.join(f'{m:>14}' for m in order))
print('-' * (14 + 14 * len(order)))
for k in ROWS:
    print(f"{k:<14}" + ''.join(
        f"{fusion_results.get(m, {}).get(k, float('nan')):>14.4f}" for m in order))
print('=' * 78)
print('Post-hoc McNemar (attention vs naive), significance:')
for name, mc in posthoc['significance'].items():
    flag = 'sig' if mc['significant_05'] else 'n.s.'
    print(f"  attention vs {name:<14}: p={mc['p_value']:.4f}  ({flag})")


In [ ]:
# ── CELL 11 ── Publication tables: results + SOTA comparison (+ CSV)
# Run after training magnifications (Cell 9) and/or the fusion model (Cell 10).

import glob, json, os
from experiments import build_results_table, build_comparison_table, write_csv

records = []

# (a) any experiments-style result JSONs
for f in glob.glob(f'{OUTPUT_PATH}/results/*.json'):
    records.append(json.load(open(f)))

# (b) the sequential per-magnification progress file
prog = f'{OUTPUT_PATH}/gpcn_vit_all_mags/training_progress.json'
if os.path.exists(prog):
    p = json.load(open(prog))
    for mag, r in p.get('completed', {}).items():
        records.append({'magnification': mag, 'backbone': 'phikon', 'use_gpcn': True,
                        'metrics': r.get('test_metrics', {})})

print(build_results_table(records))
print('\n')
print(build_comparison_table(records))
write_csv(records, f'{OUTPUT_PATH}/results/summary.csv')
print(f'\nCSV → {OUTPUT_PATH}/results/summary.csv')

In [ ]:
# ── CELL 12 ── Ablation: GPCN ON vs OFF (plain ViT) — paired over seeds, with significance
# THE key experiment for the paper: it isolates the Graph Patch Correlation
# Network's contribution. Each arm is run over multiple SEEDS and compared with a
# PAIRED t-test per metric (Delta, 95% CI, p-value). CRASH-SAFE: every (seed, arm)
# run caches its own JSON and is skipped on re-run; per-epoch checkpoints auto-resume.

import json
from experiments import run_ablation, build_results_table
from stats_tests import format_comparison_table

RESULTS_DIR = f'{OUTPUT_PATH}/results'
recs = run_ablation(mag='200X', epochs=50, seeds=(0, 1, 2),
                    data_root=DATASET_PATH, save_dir=OUTPUT_PATH,
                    results_dir=RESULTS_DIR)

# Paired significance summary written by run_ablation:
with open(f'{RESULTS_DIR}/ablation_200X_significance.json') as f:
    sig = json.load(f)
print('\nGPCN ON vs OFF @ 200X — paired t-test across seeds:')
print(format_comparison_table(sig['comparisons']))


In [ ]:
# ── CELL 13 ── Train GPCN-ViT on BACH (MULTI-CLASS: Normal / Benign / InSitu / Invasive)
#
# BACH (ICIAR-2018) Part A: 400 H&E images, 4 native classes (clinically ordered
# Normal < Benign < InSitu < Invasive). Single-magnification, so this trains the
# single-image GPCN-ViT (not the fusion model). Label-stratified image-level split
# (BACH has no patient grouping). CRASH-SAFE: auto-resumes from last_bach.pth and
# also writes per-epoch checkpoint_bach_epoch_N.pth (every save_frequency epochs)
# so you can tell which epoch was reached before a crash. Temperature-scaling
# CALIBRATION runs automatically at the end and saves a reliability diagram.

import torch
from config import get_default_config
from bach import train_bach

bach_cfg = get_default_config()
bach_cfg.data.save_dir       = OUTPUT_PATH
bach_cfg.data.batch_size     = 8
bach_cfg.training.num_epochs = 50
bach_cfg.experiment_name     = 'gpcn_bach'
bach_cfg.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

bach_result = train_bach(bach_cfg, bach_root=BACH_PATH)   # num_classes auto-detected

print('\nBACH multi-class — test metrics:')
for k, v in sorted(bach_result['test_metrics'].items()):
    if k == 'confusion_matrix':
        print(f'  {k}: {v}')
    else:
        print(f'  {k:<24} {v:.4f}')

# Calibration (temperature scaling) ran automatically at the end of training:
cal = bach_result.get('calibration')
if cal:
    print(f"\nCalibration (temperature scaling): T={cal['temperature']:.4f}")
    print(f"  ECE {cal['ece_before']:.4f} -> {cal['ece_after']:.4f}  |  "
          f"MCE {cal['mce_before']:.4f} -> {cal['mce_after']:.4f}")
    print(f"  reliability diagram: {OUTPUT_PATH}/gpcn_bach/checkpoints/bach_reliability.png")


In [ ]:
# ── CELL 13b ── BACH ablation: GPCN ON vs OFF (multi-class), paired over seeds
# Core novelty check on the 4-class task — proves the GPCN module (not just the ViT
# backbone) drives performance. Paired t-test per metric across seeds. CRASH-SAFE:
# each (seed, arm) run caches bach_ablation_seed*_gpcn*.json and is skipped on re-run.

import torch
from config import get_default_config
from bach import run_bach_ablation
from stats_tests import format_comparison_table

abl_cfg = get_default_config()
abl_cfg.data.save_dir       = OUTPUT_PATH
abl_cfg.data.batch_size     = 8
abl_cfg.training.num_epochs = 50
abl_cfg.experiment_name     = 'gpcn_bach'
abl_cfg.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

bach_abl = run_bach_ablation(abl_cfg, bach_root=BACH_PATH, seeds=(0, 1, 2),
                             results_dir=f'{OUTPUT_PATH}/results')
print('\nBACH GPCN ON vs OFF — paired t-test across seeds:')
print(format_comparison_table(bach_abl['comparisons']))


In [ ]:
# ── CELL 13c ── BACH image-level K-fold cross-validation (mean ± std + 95% CI)
# BACH Part A has no patient grouping, so this uses STRATIFIED IMAGE-LEVEL folds
# (the accepted BACH protocol). Replaces the small single split (n~80 test) with
# confidence intervals. CRASH-SAFE: each fold caches cv_bach_fold<k>.json and
# per-fold checkpoints auto-resume — just re-run this cell to continue.

import torch
from config import get_default_config
from bach import run_bach_kfold_cv
from kfold_cv import build_cv_table

bcv_cfg = get_default_config()
bcv_cfg.data.save_dir       = OUTPUT_PATH
bcv_cfg.data.batch_size     = 8
bcv_cfg.training.num_epochs = 50
bcv_cfg.experiment_name     = 'gpcn_bach'
bcv_cfg.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

bcv_out = run_bach_kfold_cv(bcv_cfg, bach_root=BACH_PATH, n_folds=5)
print('\nBACH 5-FOLD CV (mean +/- std, 95% CI):')
print(build_cv_table(bcv_out['aggregate']))


In [ ]:
# ── CELL 14 ── Patient-level K-fold cross-validation (mean ± std + 95% CI)
#
# Reviewers ask whether the headline number is stable across splits. This runs
# STRATIFIED, PATIENT-LEVEL k-fold CV (no patient in two folds -> leakage-free):
# each fold becomes the held-out TEST set, a small validation slice is carved
# from the remaining patients for model selection + decision-threshold tuning,
# and every metric is reported as mean +/- std with a 95% CI across folds.
# CRASH-SAFE: each fold caches its result (cv_fold<k>.json) and per-fold
# checkpoints auto-resume, so a Colab disconnect costs at most the in-progress
# fold -- just re-run this cell to continue.

import torch
from config import get_full_config
from kfold_cv import run_kfold_cv, build_cv_table

cv_cfg = get_full_config()
cv_cfg.data.data_root      = DATASET_PATH
cv_cfg.data.save_dir       = OUTPUT_PATH
cv_cfg.data.batch_size     = 16
cv_cfg.training.num_epochs = 50
cv_cfg.experiment_name     = 'gpcn_cv'
cv_cfg.device              = 'cuda' if torch.cuda.is_available() else 'cpu'

# Choose the protocol to cross-validate:
#   (a) a single magnification  -- the standard BreaKHis per-mag protocol
cv_cfg.data.pool_all_magnifications = False
cv_cfg.data.train_magnification     = '200X'
#   (b) magnification-agnostic  -- pool all four; uncomment to use instead:
# cv_cfg.data.pool_all_magnifications = True

cv_out = run_kfold_cv(cv_cfg, n_folds=5)   # patient-level, stratified, leakage-free

print('\n5-FOLD PATIENT-LEVEL CV (mean +/- std, 95% CI):')
print(build_cv_table(cv_out['aggregate']))
